# Pipeline quickstart: URL → canonical outputs → analysis

This notebook walks through the full singlet pipeline end-to-end:

1. **Run** the pipeline on a public SRA Run accession.
2. **Inspect** the canonical output layout (`docs/CANONICAL_OUTPUT_FORMAT.md`).
3. **Read** gene-count, USA (spliced/unspliced/ambiguous), variant, and non-host matrices on demand with the `singlet.io` + `singlet.views` API.

It assumes you have `singlet` installed (`pip install singlet`), the C++ binary on `$PATH` (or `$SINGLET_BINARY` set), and a reference bundle at `$SINGLET_REF_BASE`.

## 1. Run the pipeline

`singlet.pipeline.run` accepts an SRA accession, a URL to an `.sra`/`.1fq`/`.fastq.gz` file, a local path, or a pair of FASTQ files. It returns a `Run` describing the produced output directory.

In [ ]:
from singlet.pipeline import run

result = run(
    "SRR11537951",                # any SRR/ERR/DRR Run accession works
    output_dir="./out/SRR11537951",
    organism="human",
    threads=8,
    nonhost=True,                   # populate nonhost.json + nonhost_species.1pz
)

print("success:", result.success)
print("output_dir:", result.output_dir)
print("elapsed:", f"{result.elapsed_s:.1f}s")

Equivalent shell invocation:

```bash
singlet-process SRR11537951 \
    --output-dir ./out/SRR11537951 \
    --organism human --threads 8 --nonhost
```

Other accepted source forms:

```python
run("https://sra-pub-run-odp.s3.amazonaws.com/sra/SRR11537951/SRR11537951", "./out")
run(["R1.fastq.gz", "R2.fastq.gz"], "./out")  # paired FASTQ
run("/data/SRR11537951.1fq", "./out")          # local .1fq
```

## 2. What's in the output directory?

Every run produces the canonical per-sample layout. The minimal always-present files are:

```text
out/SRR11537951/
├── counts.1pz          # exon_body | intron_body | junctions row blocks
├── snp.1pz             # two-layer CSC (AD, DP) on population SNP panel
├── mt.1pz              # two-layer CSC (AD, DP) on chrM positions
├── cell_meta.parquet   # all cell-level scalars
├── summary.json        # all sample-level scalars + provenance
├── saturation_curve.tsv
└── star_Log.final.out
```

Optional siblings appear when feature layers are enabled: `nonhost.json` + `nonhost_species.1pz`, `guides.1pz`, `antibodies.1pz`, `vdj_gene_usage.1pz`, `donor_*`, `ambient_profile.npy`, `splice_events.tsv`.

In [ ]:
import os

for entry in sorted(os.listdir(result.output_dir)):
    path = os.path.join(result.output_dir, entry)
    size = os.path.getsize(path) / 1024
    print(f"  {entry:30s} {size:>10.1f} KB")

## 3. Open the sample

`SingletSample` is the top-level lazy reader. Everything is loaded on demand — opening a sample only reads `summary.json`.

In [ ]:
from singlet.io import SingletSample

sample = SingletSample(result.output_dir)

# summary.json — sample-level scalars + provenance
summary = sample.summary
print("protocol  :", summary.get("protocol"))
print("n_cells   :", summary.get("n_cells"))
print("map rate  :", summary.get("mapping_rate"))
print("reference :", summary.get("reference", {}).get("build_id"))

In [ ]:
# cell_meta.parquet — one row per called cell
cm = sample.cell_meta
print(cm.columns.tolist())
cm.head()

## 4. Gene-count matrix

`gene_counts(sample)` projects the three row blocks of `counts.1pz` (`exon_body`, `intron_body`, `junctions`) onto the gene axis using the splice-graph adjacency in the reference bundle's `features.fbin`. The result is the standard genes × cells sparse matrix.

In [ ]:
from singlet.views import gene_counts

X = gene_counts(sample)  # scipy.sparse.csc_matrix, shape (n_genes, n_cells)
print("shape:", X.shape, "nnz:", X.nnz)
print("per-cell total UMI (first 5):", X.sum(axis=0).A1[:5])

## 5. USA-mode (RNA velocity) decomposition

`usa(sample)` returns three matrices in the same gene × cell shape: **spliced**, **unspliced**, **ambiguous**. The decomposition uses the junction-type labels stored in the `junctions` row block of `counts.1pz`:

| Output      | Sources                                |
| ----------- | -------------------------------------- |
| `spliced`   | `exon_body` + exon-exon junctions      |
| `unspliced` | `intron_body` + exon-intron junctions  |
| `ambiguous` | intron-intron junctions                |

In [ ]:
from singlet.views import usa

trio = usa(sample)            # UsaTriplet(spliced, unspliced, ambiguous)
print("spliced  :", trio.spliced.shape, "nnz=", trio.spliced.nnz)
print("unspliced:", trio.unspliced.shape, "nnz=", trio.unspliced.nnz)
print("ambiguous:", trio.ambiguous.shape, "nnz=", trio.ambiguous.nnz)

Plug straight into scVelo:

```python
import anndata, scvelo as scv
adata = anndata.AnnData(X=trio.spliced.T)
adata.layers["spliced"] = trio.spliced.T
adata.layers["unspliced"] = trio.unspliced.T
adata.layers["ambiguous"] = trio.ambiguous.T
scv.pp.filter_and_normalize(adata)
scv.pp.moments(adata); scv.tl.velocity(adata)
```

## 6. Splice events — per-junction PSI

`psi(sample)` computes per-junction percent-spliced-in (PSI) from the `junctions` block of `counts.1pz` plus donor/acceptor adjacency from `features.fbin`. Useful for alternative-splicing QTL studies.

In [ ]:
from singlet.views import psi

P = psi(sample)              # csc_matrix, shape (n_junctions, n_cells)
print("junctions × cells:", P.shape, "defined entries:", P.nnz)

## 7. Variant tracks — SNP and mitochondrial

`snp.1pz` and `mt.1pz` use a **two-layer CSC** scheme: a single shared `indptr` + `indices`, plus two parallel data arrays (`AD` = alt-allele depth, `DP` = total depth). Reading `.vaf()` divides the two layers on the fly.

In [ ]:
snp_ad = sample.snp.ad()         # alt-allele depth
snp_dp = sample.snp.dp()         # total depth
snp_vaf = sample.snp.vaf()       # AD / DP (np.float32, masked where DP==0)
print("snp matrix shape:", snp_vaf.shape)

mt_vaf = sample.mt.vaf()         # mitochondrial heteroplasmy
print("mt matrix shape :", mt_vaf.shape)

## 8. Non-host (Kraken2 + Bracken)

When the pipeline was run with `nonhost=True`, two additional files appear:

- `nonhost.json` — raw Kraken2 report + Bracken species-level abundance.
- `nonhost_species.1pz` — sparse per-cell × species (NCBI taxid) count matrix.

No filtering or background subtraction is done at processing time — that's left to downstream analysis.

In [ ]:
if sample.nonhost is not None:
    nh = sample.nonhost
    print("top taxa by total reads:")
    species_df = nh.species_table()      # taxid, name, n_reads, abundance
    print(species_df.head(10))

    per_cell = nh.per_cell()             # csc_matrix, (n_species, n_cells)
    print("per-cell shape:", per_cell.shape, "nnz:", per_cell.nnz)

## 9. Plug into AnnData / scanpy

Round-trip into the standard ecosystem in a few lines.

In [ ]:
import anndata as ad

adata = ad.AnnData(
    X=X.T.tocsr(),                       # cells × genes
    obs=sample.cell_meta,
    var={"gene_id": sample.summary["genes"]} if isinstance(sample.summary.get("genes"), list) else None,
)
adata.layers["spliced"]   = trio.spliced.T.tocsr()
adata.layers["unspliced"] = trio.unspliced.T.tocsr()
adata.layers["ambiguous"] = trio.ambiguous.T.tocsr()
print(adata)

## Where to go next

- **Hyper-parameters**: `singlet-process --help` and [`docs/pipeline.md`](../pipeline.md) document every flag exposed by `singlet.pipeline.run`. Pass advanced binary flags through `extra_args=[...]`.
- **Format spec**: [`docs/CANONICAL_OUTPUT_FORMAT.md`](../CANONICAL_OUTPUT_FORMAT.md) is the authoritative reference for every output file.
- **Codec internals**: [`docs/format-1pz.md`](../format-1pz.md) and [`docs/format-1fq.md`](../format-1fq.md).
- **Scaling out**: see the pipeline orchestrator scripts in the singlet repo for SLURM array submission across a GEO panel.